In [ ]:
"""
=============================================================================
 ÉTUDE 1 — ABLATION : CONTRIBUTION DES MODULES M1 / M3 / M4
=============================================================================
 Objectif : quantifier l'apport de chaque module et comparer les formules
            de fusion successives (uniforme, V3, V4, V5).
 Entrée   : resultats_phase4_v5.csv
 Sortie   : tableau AUC + figure comparative
 GPU      : non nécessaire
=============================================================================
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION CENTRALE
# ============================================================
CSV_PATH = '/kaggle/input/datasets/abdallahouahmane/resultas-phase4/resultats_phase4_v5.csv'

# Pondérations de la formule déployée (V5)
W = {'ctr': 0.15, 'cong': 0.40, 'calcif': 0.10, 'cardio': 0.35}

NOM_V5 = 'Pipeline V5\n(déployé)'
NOM_V4 = 'Optimum V4\n(M1+M3+M4)'
NOM_V3 = 'Pipeline V3\n(M1+M4)'


def score_v5(d):
    return (W['ctr'] * d['s_ctr'] + W['cong'] * d['s_cong']
            + W['calcif'] * d['s_calcif'] + W['cardio'] * d['prob_cardio'])


# ============================================================
# 1. CHARGEMENT
# ============================================================
print("=" * 62)
print("  ÉTUDE 1 — ABLATION DES MODULES")
print("=" * 62)

df = pd.read_csv(CSV_PATH)
print(f"\n  CSV chargé : {len(df)} images")

colonnes_requises = ['label_reel', 's_ctr', 's_calcif', 's_cong',
                     'prob_cardio', 'risk_global']
manquantes = [c for c in colonnes_requises if c not in df.columns]
if manquantes:
    raise SystemExit(f"Colonnes manquantes : {manquantes}")

df['y_true'] = df['label_reel'].apply(
    lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0)

print(f"  Positifs : {df['y_true'].sum()} | "
      f"Négatifs : {(df['y_true'] == 0).sum()}")

# Contrôle : la formule recalculée doit correspondre au CSV
ecart = (score_v5(df) * 100 - df['risk_global']).abs().max()
print(f"  Contrôle formule V5 vs risk_global : écart max = {ecart:.3f} point")

# ============================================================
# 2. CONFIGURATIONS
# ============================================================
configurations = {

    # ── Composants isolés ─────────────────────────────────
    'CTR seul\n(M1)': (
        df['s_ctr'],
        'Score = s_ctr'),
    'Calcif seul\n(M3)': (
        df['s_calcif'],
        'Score = s_calcif'),
    'Congestion seule\n(M4 œdème)': (
        df['s_cong'],
        'Score = s_cong = prob_edema'),
    'Cardiomégalie seule\n(M4 cardio)': (
        df['prob_cardio'],
        'Score = prob_cardio'),

    # ── Ablations : V5 privé d'un module ──────────────────
    'V5 sans CTR': (
        (W['cong'] * df['s_cong'] + W['calcif'] * df['s_calcif']
         + W['cardio'] * df['prob_cardio']) / (1 - W['ctr']),
        'V5 sans s_ctr, poids renormalisés'),
    'V5 sans Calcif': (
        (W['ctr'] * df['s_ctr'] + W['cong'] * df['s_cong']
         + W['cardio'] * df['prob_cardio']) / (1 - W['calcif']),
        'V5 sans s_calcif, poids renormalisés'),
    'V5 sans Congestion': (
        (W['ctr'] * df['s_ctr'] + W['calcif'] * df['s_calcif']
         + W['cardio'] * df['prob_cardio']) / (1 - W['cong']),
        'V5 sans s_cong, poids renormalisés'),
    'V5 sans prob_cardio': (
        (W['ctr'] * df['s_ctr'] + W['cong'] * df['s_cong']
         + W['calcif'] * df['s_calcif']) / (1 - W['cardio']),
        'V5 sans prob_cardio, poids renormalisés'),

    # ── Formules successives ──────────────────────────────
    'Fusion uniforme\n(50/25/25)': (
        0.50 * df['s_ctr'] + 0.25 * df['s_calcif'] + 0.25 * df['s_cong'],
        'Pondération a priori'),
    NOM_V3: (
        0.45 * df['s_ctr'] + 0.55 * df['s_cong'],
        '0.45×s_ctr + 0.55×s_cong'),
    NOM_V4: (
        0.35 * df['s_ctr'] + 0.45 * df['s_cong'] + 0.20 * df['s_calcif'],
        '0.35×s_ctr + 0.45×s_cong + 0.20×s_calcif'),
    NOM_V5: (
        score_v5(df),
        f"{W['ctr']}×s_ctr + {W['cong']}×s_cong + "
        f"{W['calcif']}×s_calcif + {W['cardio']}×prob_cardio"),
}

# ============================================================
# 3. CALCUL DES AUC
# ============================================================
print(f"\n{'='*62}")
print("  AUC PAR CONFIGURATION")
print(f"{'='*62}\n")
print(f"  {'Configuration':<32} {'AUC':>8} {'Δ vs V5':>10}")
print("  " + "-" * 52)

resultats = {}
for nom, (scores, description) in configurations.items():
    resultats[nom] = {'auc': round(roc_auc_score(df['y_true'], scores), 4),
                      'scores': scores, 'description': description}

auc_v5 = resultats[NOM_V5]['auc']
auc_v4 = resultats[NOM_V4]['auc']
auc_v3 = resultats[NOM_V3]['auc']

for nom, v in resultats.items():
    delta = v['auc'] - auc_v5
    marque = '  ← DÉPLOYÉ' if nom == NOM_V5 else ''
    print(f"  {nom.replace(chr(10), ' '):<32} {v['auc']:>8.4f} "
          f"{delta:>+10.4f}{marque}")

# ============================================================
# 4. ANALYSE
# ============================================================
print(f"\n{'='*62}")
print("  CONTRIBUTION DE CHAQUE MODULE")
print(f"{'='*62}\n")
print(f"  {'Module':<16} {'AUC seul':>10} {'V5 sans lui':>13} {'Apport':>9}")
print("  " + "-" * 50)

paires = [
    ('CTR (M1)',      'CTR seul\n(M1)',                 'V5 sans CTR'),
    ('Calcif (M3)',   'Calcif seul\n(M3)',              'V5 sans Calcif'),
    ('Œdème (M4)',    'Congestion seule\n(M4 œdème)',   'V5 sans Congestion'),
    ('Cardio (M4)',   'Cardiomégalie seule\n(M4 cardio)', 'V5 sans prob_cardio'),
]
for libelle, cle_seul, cle_sans in paires:
    a_seul = resultats[cle_seul]['auc']
    a_sans = resultats[cle_sans]['auc']
    print(f"  {libelle:<16} {a_seul:>10.4f} {a_sans:>13.4f} "
          f"{auc_v5 - a_sans:>+9.4f}")

print(f"\n  Évolution des formules de fusion :")
print(f"    Fusion uniforme  : {resultats['Fusion uniforme\n(50/25/25)']['auc']:.4f}")
print(f"    Pipeline V3      : {auc_v3:.4f}")
print(f"    Optimum V4       : {auc_v4:.4f}  ({auc_v4 - auc_v3:+.4f} vs V3)")
print(f"    Pipeline V5      : {auc_v5:.4f}  ({auc_v5 - auc_v3:+.4f} vs V3)")

composants = [resultats[c]['auc'] for c in
              ['CTR seul\n(M1)', 'Calcif seul\n(M3)',
               'Congestion seule\n(M4 œdème)', 'Cardiomégalie seule\n(M4 cardio)']]
meilleur_comp = max(composants)
print(f"\n  Meilleur composant isolé : {meilleur_comp:.4f}")
print(f"  Gain de la fusion V5     : {auc_v5 - meilleur_comp:+.4f}")

# ============================================================
# 5. AUC PAR PATHOLOGIE
# ============================================================
print(f"\n{'='*62}")
print("  AUC PAR PATHOLOGIE — V3 vs V5")
print(f"{'='*62}\n")

y_cardio = df['label_reel'].str.contains('Cardiomegaly', na=False).astype(int)
y_edema  = df['label_reel'].str.contains('Edema', na=False).astype(int)

print(f"  {'Pathologie':<16} {'V3':>10} {'V5':>10} {'Gain':>9}")
print("  " + "-" * 47)
for libelle, y_bin in [('Cardiomégalie', y_cardio), ('Œdème', y_edema)]:
    a3 = roc_auc_score(y_bin, resultats[NOM_V3]['scores'])
    a5 = roc_auc_score(y_bin, resultats[NOM_V5]['scores'])
    print(f"  {libelle:<16} {a3:>10.4f} {a5:>10.4f} {a5 - a3:>+9.4f}")

# ============================================================
# 6. SCORES MOYENS PAR GROUPE
# ============================================================
print(f"\n{'='*62}")
print("  SCORES MOYENS PAR GROUPE")
print(f"{'='*62}\n")

groupes = {
    'Normal':        df[df['label_reel'] == 'No Finding'],
    'Cardiomégalie': df[df['label_reel'].str.contains('Cardiomegaly', na=False)],
    'Œdème':         df[df['label_reel'].str.contains('Edema', na=False)],
}
print(f"  {'Groupe':<16} {'s_ctr':>8} {'s_calcif':>10} {'s_cong':>8} "
      f"{'p_cardio':>10} {'score':>8}")
print("  " + "-" * 62)
for nom, sub in groupes.items():
    if len(sub):
        print(f"  {nom:<16} {sub['s_ctr'].mean():>8.3f} "
              f"{sub['s_calcif'].mean():>10.3f} {sub['s_cong'].mean():>8.3f} "
              f"{sub['prob_cardio'].mean():>10.3f} "
              f"{sub['risk_global'].mean():>7.1f}%")

# ============================================================
# 7. VISUALISATION
# ============================================================
fig = plt.figure(figsize=(22, 13))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

noms = [n.replace('\n', ' ') for n in resultats.keys()]
aucs = [v['auc'] for v in resultats.values()]

couleurs = []
for nom in resultats.keys():
    if nom == NOM_V5:
        couleurs.append('#C0392B')      # rouge — formule déployée
    elif 'V3' in nom or 'V4' in nom or 'uniforme' in nom:
        couleurs.append('#1B2A4A')      # marine — formules antérieures
    elif 'seul' in nom or 'seule' in nom:
        couleurs.append('#E67E22')      # orange — composants isolés
    else:
        couleurs.append('#7F8C8D')      # gris — ablations

# ── Panel 1 : barres AUC ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.barh(noms, aucs, color=couleurs, alpha=0.88, edgecolor='white')
ax1.axvline(auc_v5, color='#C0392B', ls='--', lw=2,
            label=f'Pipeline V5 (AUC = {auc_v5:.4f})')
ax1.axvline(0.50, color='gray', ls=':', lw=1, alpha=0.6, label='Hasard')
ax1.set_xlim(0.45, 0.98)
ax1.set_xlabel('AUC-ROC', fontsize=11)
ax1.set_title('Étude 1 — AUC par configuration\n'
              'Rouge : formule déployée | Marine : formules antérieures | '
              'Orange : composants isolés | Gris : ablations',
              fontweight='bold', fontsize=11)
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(alpha=0.3, axis='x')
for bar, a in zip(bars, aucs):
    ax1.text(a + 0.004, bar.get_y() + bar.get_height() / 2,
             f'{a:.4f}', va='center', fontsize=9, fontweight='bold')

# ── Panel 2 : composants isolés ───────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
libelles = ['CTR\n(M1)', 'Calcif\n(M3)', 'Œdème\n(M4)', 'Cardio\n(M4)']
bars2 = ax2.bar(libelles, composants,
                color=['#1D9E75', '#B45309', '#185FA5', '#8E44AD'], alpha=0.85)
ax2.axhline(auc_v5, color='#C0392B', ls='--', lw=2,
            label=f'V5 = {auc_v5:.3f}')
ax2.axhline(0.50, color='gray', ls=':', alpha=0.6)
ax2.set_ylim(0.40, 1.0)
ax2.set_ylabel('AUC-ROC')
ax2.set_title('Composants isolés\nvs fusion V5', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, axis='y')
for bar, a in zip(bars2, composants):
    ax2.text(bar.get_x() + bar.get_width() / 2, a + 0.008,
             f'{a:.3f}', ha='center', fontsize=10, fontweight='bold')

# ── Panel 3 : courbes ROC ─────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
courbes = {
    'CTR seul\n(M1)':                  ('#1D9E75', '--', 1.5),
    'Calcif seul\n(M3)':               ('#B45309', '--', 1.5),
    'Congestion seule\n(M4 œdème)':    ('#185FA5', '--', 1.5),
    NOM_V3:                            ('#1B2A4A', '-',  2.0),
    NOM_V5:                            ('#C0392B', '-',  3.2),
}
for nom, (col, ls, lw) in courbes.items():
    fpr, tpr, _ = roc_curve(df['y_true'], resultats[nom]['scores'])
    ax3.plot(fpr, tpr, color=col, ls=ls, lw=lw,
             label=f"{nom.replace(chr(10), ' ')} ({resultats[nom]['auc']:.3f})")
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax3.set_xlabel('1 − Spécificité')
ax3.set_ylabel('Sensibilité')
ax3.set_title('Courbes ROC', fontweight='bold')
ax3.legend(fontsize=7, loc='lower right')
ax3.grid(alpha=0.3)

# ── Panel 4 : distribution des scores ─────────────────────
ax4 = fig.add_subplot(gs[1, 1])
donnees = [g['risk_global'].values for g in groupes.values()]
bp = ax4.boxplot(donnees, tick_labels=list(groupes.keys()),
                 patch_artist=True, notch=True)
for patch, c in zip(bp['boxes'], ['#2ECC71', '#E74C3C', '#F39C12']):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax4.axhline(35, color='#C0392B', ls='--', lw=1.8, label='Seuil ÉLEVÉ (35 %)')
ax4.set_ylabel('risk_global (%)')
ax4.set_title('Distribution du score V5\npar groupe', fontweight='bold')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3, axis='y')

# ── Panel 5 : tableau ─────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
lignes = [[n.replace('\n', ' '), f"{v['auc']:.4f}", f"{v['auc'] - auc_v5:+.4f}"]
          for n, v in resultats.items()]
tab = ax5.table(cellText=lignes,
                colLabels=['Configuration', 'AUC', 'Δ vs V5'],
                cellLoc='center', loc='center')
tab.auto_set_font_size(False)
tab.set_fontsize(7)
tab.scale(1, 1.5)
ax5.set_title('Récapitulatif', fontweight='bold', pad=18)

plt.suptitle(
    f'ÉTUDE 1 — Ablation des modules | Pipeline V5 : AUC = {auc_v5:.4f} | '
    f'n = {len(df)} images NIH',
    fontsize=14, fontweight='bold')

for ext in ('png', 'svg'):
    plt.savefig(f'etude1_ablation_v5.{ext}', dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

# ============================================================
# 8. SYNTHÈSE
# ============================================================
print(f"\n{'='*62}")
print("  SYNTHÈSE — ÉTUDE 1")
print(f"{'='*62}\n")

for rang, (nom, v) in enumerate(
        sorted(resultats.items(), key=lambda x: x[1]['auc'], reverse=True), 1):
    marque = '  ← DÉPLOYÉ' if nom == NOM_V5 else ''
    print(f"  {rang:>2}. {nom.replace(chr(10), ' '):<34} "
          f"{v['auc']:.4f}{marque}")

print(f"\n  Le retrait de chaque module dégrade l'AUC, ce qui établit")
print(f"  que les quatre composantes contribuent au score final.")
print(f"\n  Fichiers générés : etude1_ablation_v5.png / .svg")